# Findings summary — full Final-results analysis

Reads **every file** under `Final-results/Final/`, aggregates all 63 models, and writes:

- `Final-results/Final/findings/master_metrics.csv`
- `Final-results/Final/findings/FINDINGS.md`
- `Final-results/Final/findings/plots/*.png`

Run after `hourly.ipynb`, `15min.ipynb`, and `behavior_analysis.ipynb`.

**Parts:** 0 inventory → 1 performance → 2 SHAP → 3 IG → 4 erasure → 5 fidelity → 6 cross-findings → 7 report

In [ ]:
# Local clone (default) or Colab Drive
from pathlib import Path

if Path("../Final-results/Final").exists():
    RESULTS_ROOT = Path("../Final-results/Final").resolve()
elif Path("/content/drive/MyDrive/Shared-Colab-Storage/Final").exists():
    RESULTS_ROOT = Path("/content/drive/MyDrive/Shared-Colab-Storage/Final")
else:
    RESULTS_ROOT = Path("../Final-results/Final").resolve()

FINDINGS_DIR = RESULTS_ROOT / "findings"
print("RESULTS_ROOT:", RESULTS_ROOT)

## Setup

In [ ]:
!pip install -q pandas numpy matplotlib seaborn scipy

import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "findings_runner.py").exists():
    sys.path.insert(0, str(NOTEBOOK_DIR))
elif (NOTEBOOK_DIR / "Final" / "findings_runner.py").exists():
    sys.path.insert(0, str(NOTEBOOK_DIR / "Final"))

import findings_runner as fr
import pandas as pd
from IPython.display import display, Markdown

---
## Part 0 — File inventory

Walk every file under the results tree. Nothing skipped.

In [ ]:
FINDINGS_DIR.mkdir(parents=True, exist_ok=True)
(FINDINGS_DIR / "plots").mkdir(exist_ok=True)

manifest, gaps = fr.part0_inventory.__wrapped__() if hasattr(fr.part0_inventory, '__wrapped__') else None

# run via main's inventory (patch RESULTS_ROOT first)
fr.RESULTS_ROOT = RESULTS_ROOT
fr.FINDINGS_DIR = FINDINGS_DIR
fr.PLOTS_DIR = FINDINGS_DIR / "plots"

manifest, gaps = fr.part0_inventory()
print("Total files:", len(manifest))
print("Missing behavior CSVs:", len(gaps))
display(manifest.groupby(["ext"]).size().rename("count"))
display(manifest.groupby(["category"]).size().rename("count").head(15))

**Summary:** Complete inventory saved to `findings/file_manifest.csv`. Any missing expected behavior file is in `gaps_report.csv`.

---
## Part 1 — Prediction performance (all models)

RMSE / R² for every stack × window. Best models marked from `analysis/best_models.json`.

In [ ]:
perf = fr.load_metrics()
shap_df = fr.load_shap_all()
ig_df = fr.load_ig_all()
eras_df = fr.load_erasure_all()
fid_df = fr.load_fidelity_all()
master = fr.build_master(perf, shap_df, ig_df, eras_df, fid_df)
master.to_csv(FINDINGS_DIR / "master_metrics.csv", index=False)

print("Models in master table:", len(master))
display(master.sort_values("rmse_kwh").head(10)[["track","stack","window","rmse_kwh","r2","is_best_track"]])

---
## Part 2 — SHAP (all 63 models)

Which features drive predictions? Check how often `Usage_kWh` ranks first.

In [ ]:
usage_pct = master.groupby(["track", "stack"])["usage_kwh_rank1"].mean() * 100
print("% models with Usage_kWh as top SHAP feature:")
print(usage_pct.round(1))
print("\nTop features frequency (all models):")
print(master["top1_feature"].value_counts().head(8))

---
## Part 3 — IG memory patterns (all 63 models)

Derived metrics: `recency_ratio`, `start_ratio`, `u_shape_score`, `ig_pattern`.

- **recency_dominant:** single/double — spike at recent steps
- **u_shaped:** bidir — start + end matter, middle flat

In [ ]:
print("IG pattern counts:")
display(master.groupby(["stack", "ig_pattern"]).size().unstack(fill_value=0))
print("\nMean recency_ratio by stack:")
print(master.groupby("stack")["recency_ratio"].mean().round(3))
print("\nMean u_shape_score by stack:")
print(master.groupby("stack")["u_shape_score"].mean().round(2))

---
## Part 4 — Memory erasure (all 63 models)

RMSE change when oldest 25/50/75% of window is replaced with training mean.

In [ ]:
print("Mean delta_25 (RMSE increase at 25% erase) by stack:")
print(master.groupby("stack")["delta_25"].mean().round(2))
display(master.groupby("track")[["delta_25","delta_50","delta_75"]].mean().round(2))

---
## Part 5 — Fidelity (all 63 models)

Zero top SHAP features — does RMSE jump? Validates explanations.

In [ ]:
print("Top faithful feature counts:")
print(master["top_faithful_feature"].value_counts().head(6))
print("\nMean max_delta by stack:", master.groupby("stack")["max_delta"].mean().round(2).to_dict())

---
## Part 6 — Cross-cutting plots (all models)

Aggregate visualizations saved to `findings/plots/`.

In [ ]:
fr.plot_all_figures(master)
plots = sorted((FINDINGS_DIR / "plots").glob("*.png"))
print(f"Saved {len(plots)} plots:")
for p in plots:
    print(" ", p.name)

---
## Part 7 — Final report

Auto-written conclusions with population-level statistics.

In [ ]:
lime = fr.load_lime_spearman()
if len(lime) > 0:
    lime.to_csv(FINDINGS_DIR / "lime_spearman_subset.csv", index=False)

report = fr.write_findings_md(master, manifest, gaps)
display(Markdown(report[:3500] + "\n\n... (see FINDINGS.md for full text)"))

## Done

Open `Final-results/Final/findings/FINDINGS.md` for the full report.

Key files for presentation:
- `plots/ig_compare_win24_hourly.png` — architecture behavior
- `plots/recency_ratio_by_stack.png`
- `plots/ig_erasure_scatter.png`
- `master_metrics.csv` — all numbers in one table

In [ ]:
# One-shot: re-run everything
# fr.main(RESULTS_ROOT)